# GPUDirect Storage (GDS)

A practical reference for **GPUDirect Storage (GDS)** — NVIDIA's technology for a
direct DMA data path between storage (local NVMe or remote NVMe-oF / parallel
filesystems) and GPU memory, bypassing the CPU bounce buffer. GDS is part of the
**NVIDIA Magnum IO** stack and is consumed through the **cuFile** API
(`libcufile`).


## Table of Contents

1. [Introduction](#introduction)
2. [Key Features](#key-features)
3. [Architecture Overview](#architecture)
4. [Installation](#installation)
5. [Basic Usage](#basic-usage)
6. [Advanced Features](#advanced-features)
7. [Use Cases](#use-cases)
8. [Best Practices](#best-practices)
9. [Common Pitfalls](#pitfalls)
10. [Performance Optimization](#performance)
11. [Production Deployment](#deployment)
12. [Monitoring and Observability](#monitoring)
13. [Troubleshooting](#troubleshooting)
14. [Comparison with Alternatives](#comparison)
15. [Resources](#resources)


## Introduction
<a id="introduction"></a>

As GPUs got faster, the **storage-to-GPU data path** became the bottleneck for
data-hungry workloads. In the traditional path, data read from an NVMe drive
lands in a **bounce buffer in CPU system memory** first, and is then copied a
second time across PCIe into GPU memory. That extra hop burns CPU cycles,
consumes system-memory bandwidth, and adds latency.

### What is it?

GPUDirect Storage creates a **direct memory access (DMA) path** between a storage
device (or NIC for remote storage) and GPU memory. The DMA engine moves data
straight into `cudaMalloc`'d GPU buffers without staging through CPU system
memory. The CPU only orchestrates the transfer (issues the I/O); it never touches
the bytes. GDS is the storage member of the GPUDirect family, alongside
GPUDirect RDMA (NIC↔GPU) and GPUDirect P2P (GPU↔GPU).

You program against it through the **cuFile API** in `libcufile` (e.g.
`cuFileRead` / `cuFileWrite`), and the **`nvidia-fs`** kernel module brokers the
DMA between the filesystem/block layer and the GPU's BAR1 memory.

### Why use it?

- **Higher effective bandwidth** — saturate PCIe and many NVMe drives in
  parallel instead of being capped by a single CPU memcpy.
- **Lower latency** — one DMA hop instead of two copies.
- **Lower CPU utilization** — frees cores that would otherwise spin on `memcpy`,
  leaving them for data preprocessing and augmentation.
- **Less system-memory bandwidth pressure** — bytes never traverse DDR, so the
  CPU's memory subsystem isn't a shared bottleneck.
- **Scales with drives and GPUs** — throughput grows as you add NVMe devices or
  use a parallel filesystem over RDMA (Lustre, GPFS, WekaFS, VAST, DDN).

### When to use it?

GDS pays off when I/O is large, sequential-ish, and on the critical path:

- **Deep-learning input pipelines** over very large datasets that don't fit in
  the OS page cache (large-image, video, genomics, point-cloud training).
- **Checkpoint save/restore** for large models — writing/reading multi-GB tensors
  directly from GPU memory.
- **GPU-accelerated analytics / ETL** — RAPIDS `cuDF` reading Parquet/ORC, or
  vector-database index loads.
- **HPC and inference cold-starts** where weights are streamed from fast storage
  into GPU memory.

It is **not** worth it for tiny, random, latency-insensitive reads, or when the
working set comfortably fits in host RAM / page cache.


## Key Features
<a id="key-features"></a>

### Core Capabilities of GPUDirect Storage

| Feature | Description | Benefit |
|---------|-------------|----------|
| Direct DMA to GPU memory | DMA engine writes storage data straight into `cudaMalloc` buffers, skipping the CPU bounce buffer | Removes a full memory copy and CPU involvement from the I/O path |
| cuFile API (`libcufile`) | POSIX-like `cuFileRead`/`cuFileWrite` calls on registered file handles and GPU buffers | Familiar, explicit programming model that frameworks build on |
| `nvidia-fs` kernel module | Kernel driver that pins GPU BAR1 memory and arranges peer-to-peer DMA with the storage/NIC | Enables true GDS on supported filesystems and devices |
| Local + remote storage | Works with local NVMe and remote NVMe-oF / parallel FS over **GPUDirect RDMA** | One API for direct-attached and networked storage |
| Compatibility (POSIX) mode | Transparent fallback through a CPU bounce buffer when true GDS isn't available | Code keeps working on unsupported paths, with reduced benefit |
| Buffer & handle registration | `cuFileBufRegister` / `cuFileHandleRegister` pre-pin memory and prepare DMA mappings | Amortizes setup cost; near-peak bandwidth on hot buffers |
| Async & batch I/O | `cuFileReadAsync` / `cuFileBatchIOSubmit` stream-ordered and batched submissions | Overlaps I/O with compute and hides per-call overhead |


## Architecture Overview
<a id="architecture"></a>

GDS replaces the two-copy path (storage → CPU sysmem bounce buffer → GPU) with a
single DMA hop (storage/NIC → GPU memory) over PCIe peer-to-peer.

```
TRADITIONAL PATH (no GDS)                 GPUDirect STORAGE PATH
-------------------------                 ----------------------
 +---------+                               +---------+
 |  NVMe / |                               |  NVMe / |
 |  NIC    |                               |  NIC    |
 +----+----+                               +----+----+
      | DMA                                     | DMA (peer-to-peer)
      v                                         |   over PCIe switch
 +---------+   memcpy   +---------+             |
 | CPU sys |==========> |  GPU    |             v
 | memory  |  (PCIe)    | memory  |        +---------+
 +---------+            +---------+        |  GPU    |
   ^   CPU touches bytes twice            |  memory  |
   |   (bounce buffer + copy)            +---------+
                                          CPU only issues the I/O;
                                          bytes never enter sysmem
```

### Components

1. **cuFile API / `libcufile`** — user-space library exposing `cuFileRead`,
   `cuFileWrite`, handle/buffer registration, async and batch submission. This is
   what applications and frameworks (DALI, cuDF, kvikio, PyTorch storage) call.
2. **`nvidia-fs` kernel module** — pins GPU BAR1 memory, validates O_DIRECT
   alignment, and programs the DMA so the storage stack writes/reads GPU memory
   directly. Installed via DKMS alongside the NVIDIA driver.
3. **Storage / transport layer** — local NVMe block devices, or remote storage
   over **NVMe-oF / RDMA** (RoCE or InfiniBand) using MLNX_OFED. Parallel
   filesystems (Lustre/EXAScaler, GPFS, WekaFS, VAST, BeeGFS) ship GDS-aware
   clients.
4. **`/etc/cufile.json`** — runtime configuration: compat-mode policy, RDMA
   device lists, max direct I/O size, cache sizes, and logging.
5. **GPU + PCIe topology** — a PCIe switch (or the CPU root complex) that allows
   peer-to-peer DMA between the storage/NIC endpoint and the GPU. Topology
   determines achievable bandwidth.


## Installation
<a id="installation"></a>

### Prerequisites

- A **data-center / supported NVIDIA GPU** (e.g. A100, H100; many RTX/Quadro and
  consumer GPUs run only in compatibility mode).
- **NVIDIA data-center driver** matching the CUDA toolkit.
- **CUDA Toolkit 11.4+** (GDS shipped GA with 11.4; newer toolkits recommended).
- A **GDS-supported filesystem** on `O_DIRECT`-capable storage: local **ext4** or
  **XFS** on NVMe, or a parallel FS (Lustre/EXAScaler, GPFS, WekaFS, VAST,
  BeeGFS).
- For **remote storage**: **MLNX_OFED / DOCA-OFED** and an RDMA-capable NIC
  (ConnectX) for NVMe-oF over RoCE/InfiniBand.
- The **`nvidia-fs`** kernel module (DKMS) and the **`gds`** packages.

### Installation Steps

GDS installs as part of the CUDA toolkit metapackage, or standalone:

```bash
# Ubuntu/Debian — install GDS with the CUDA repo configured
sudo apt-get update
sudo apt-get install -y nvidia-gds            # pulls libcufile + nvidia-fs DKMS
# or pin to a toolkit version, e.g. nvidia-gds-12-4

# Verify the nvidia-fs kernel module is built and loaded
sudo modprobe nvidia-fs
lsmod | grep nvidia_fs

# Platform self-check: confirms driver, nvidia-fs, filesystems, and GPU support
/usr/local/cuda/gds/tools/gdscheck.py -p
```

`gdscheck -p` is the single most useful command after install: it prints whether
GDS is supported, which filesystems are in supported/unsupported/compat-only
state, RDMA status, and the active `cufile.json`.

**Note**: The cell below is illustrative shell, not Python. Uncomment and run it
in a terminal on a GDS-capable host (it will not work in Colab).


In [ ]:
# Shell commands — run in a terminal on a GDS-capable host, not in Colab.
#
# !sudo apt-get install -y nvidia-gds
# !sudo modprobe nvidia-fs
# !/usr/local/cuda/gds/tools/gdscheck.py -p     # platform/support check
# !cat /etc/cufile.json                          # active GDS configuration
# !lsmod | grep nvidia_fs                         # confirm kernel module loaded
print("Run gdscheck.py -p on a GDS host to confirm support before benchmarking.")


## Basic Usage
<a id="basic-usage"></a>

### Quick Start Example

The easiest way to use GDS from Python is **`kvikio`** (RAPIDS), which wraps the
cuFile API and reads/writes directly into GPU device memory (CuPy / RAPIDS
arrays). The pattern mirrors the C cuFile flow: open a `CuFile`, then
`read`/`write` against a device buffer at a given file offset.


In [ ]:
# Python GDS read/write with kvikio (pip install kvikio-cu12 cupy-cuda12x)
# Falls back to compatibility mode automatically if true GDS is unavailable.
import cupy as cp
from kvikio import CuFile

n = 64 * 1024 * 1024  # 64 MiB, O_DIRECT-friendly size
src = cp.arange(n, dtype=cp.uint8)            # data already in GPU memory
dst = cp.empty_like(src)

# WRITE: GPU memory -> file, direct DMA (no CPU bounce buffer)
with CuFile("/mnt/nvme/gds_demo.bin", "w") as f:
    nbytes = f.write(src)
    print(f"wrote {nbytes} bytes from device memory")

# READ: file -> GPU memory, direct DMA
with CuFile("/mnt/nvme/gds_demo.bin", "r") as f:
    nbytes = f.read(dst)
    print(f"read {nbytes} bytes into device memory")

assert bool((src == dst).all()), "round-trip mismatch"
print("GDS round-trip verified entirely in GPU memory")


The underlying C flow that `kvikio` performs looks like this — useful to know
when reading framework code or NVIDIA samples:

```c
cuFileDriverOpen();                                   // init the GDS driver
CUfileDescr_t descr = { .handle.fd = fd,
                        .type = CU_FILE_HANDLE_TYPE_OPAQUE_FD };
cuFileHandleRegister(&handle, &descr);                // register the file handle
cuFileBufRegister(devPtr, size, 0);                   // pin/register GPU buffer
cuFileRead(handle, devPtr, size, fileOffset, 0);      // DMA straight into GPU mem
cuFileBufDeregister(devPtr);
cuFileHandleDeregister(handle);
cuFileDriverClose();
```

Files must be opened with **`O_DIRECT`**, and offsets/sizes/device pointers must
meet the device's DMA alignment (typically **4 KiB**). `kvikio` handles this for
you; the raw C API does not.


## Advanced Features
<a id="advanced-features"></a>

### Buffer/handle registration, async, and batch I/O

- **Pre-registration** — `cuFileBufRegister` / `cuFileHandleRegister` pin GPU
  memory and set up DMA mappings once, so repeated I/O on the same buffer avoids
  per-call pinning overhead. Reuse a small pool of registered buffers in a hot
  loop rather than registering each transfer.
- **Asynchronous, stream-ordered I/O** — `cuFileReadAsync` / `cuFileWriteAsync`
  enqueue transfers on a CUDA stream so storage I/O overlaps with kernels. This
  is the key to hiding I/O latency in a training step.
- **Batch I/O** — `cuFileBatchIOSetUp` / `cuFileBatchIOSubmit` submit many
  small/medium I/Os in one call, amortizing submission overhead — ideal for
  scatter/gather reads from a parallel filesystem.
- **Compatibility mode** — when true GDS isn't available, `libcufile`
  transparently routes through a CPU bounce buffer so the same code runs
  everywhere (with reduced benefit). Controlled by
  `properties.allow_compat_mode` in `cufile.json`.


In [ ]:
# Async, stream-overlapped GDS reads with kvikio: I/O futures overlap with compute.
import cupy as cp
from kvikio import CuFile

chunk = 16 * 1024 * 1024          # 16 MiB chunks
buffers = [cp.empty(chunk, dtype=cp.uint8) for _ in range(4)]
stream = cp.cuda.Stream()

with CuFile("/mnt/nvme/gds_demo.bin", "r") as f:
    futures = []
    with stream:
        for i, buf in enumerate(buffers):
            # Non-blocking: returns an IOFuture; DMA proceeds on the stream
            futures.append(f.pread(buf, file_offset=i * chunk))
    # Overlap other GPU work here while DMA is in flight ...
    total = sum(fut.get() for fut in futures)   # synchronize and collect counts
print(f"async-read {total} bytes across {len(buffers)} overlapped chunks")


## Use Cases
<a id="usecases"></a>

### Real-world Applications of GPUDirect Storage

#### Use Case 1: Large-dataset deep-learning input pipelines

- **Context**: Training on datasets far larger than host RAM (high-res imagery,
  video, genomics, LiDAR). The page cache can't hold the working set, so every
  epoch streams from storage.
- **Implementation**: **NVIDIA DALI** with its GDS-backed reader, or `kvikio`
  decoding directly into GPU memory. Data lands in GPU buffers ready for
  augmentation, freeing CPU cores for the rest of the pipeline.
- **Results**: Higher sustained samples/sec and dramatically lower CPU load,
  removing the classic "CPU-bound data loader" bottleneck.

#### Use Case 2: Large-model checkpointing

- **Context**: Saving/restoring multi-GB or multi-TB model + optimizer state
  during long training runs, where checkpoint stalls waste GPU time.
- **Implementation**: Write tensors straight from GPU memory to a parallel FS over
  RDMA via cuFile; restore by DMA-ing weights back into device memory on resume.
- **Results**: Shorter checkpoint windows and faster restart after pre-emption or
  failure, improving effective GPU utilization across the run.

#### Use Case 3: GPU-accelerated analytics / ETL

- **Context**: RAPIDS `cuDF` reading large Parquet/ORC files, or loading vector
  / ANN indexes into GPU memory.
- **Implementation**: `cudf.read_parquet` and kvikio use cuFile under the hood to
  pull columnar data directly into device buffers.
- **Results**: Faster query/ingest startup and less CPU contention when many
  GPUs read shared storage concurrently.


## Best Practices
<a id="best-practices"></a>

### Recommended Practices for GPUDirect Storage

1. **Verify support before optimizing** — run `gdscheck.py -p` and confirm your
   filesystem/device shows as *Supported*, not *compat-only*. Benchmarking on a
   compat-mode path measures the bounce buffer, not GDS.
2. **Honor O_DIRECT alignment** — keep file offsets, transfer sizes, and device
   pointers 4 KiB-aligned (or the device's reported `MAX_DIRECT_IO_SIZE`
   granularity). Misalignment silently falls back to compat mode or fails.
3. **Use large, sequential-ish I/O** — favor multi-MB transfers; tiny random
   reads don't amortize DMA setup. Use **batch I/O** when you must do many small
   reads.
4. **Register and reuse buffers** — pre-register a pool of GPU buffers with
   `cuFileBufRegister` and recycle them in hot loops instead of registering per
   transfer.
5. **Overlap I/O with compute** — use async/stream-ordered cuFile calls so DMA
   runs concurrently with kernels; never block the training step on synchronous
   reads.
6. **Mind PCIe topology** — place the GPU and the NVMe/NIC under the same PCIe
   switch (peer-to-peer) for best bandwidth; cross-socket P2P over the CPU root
   complex is much slower.
7. **Tune `cufile.json` deliberately** — set RDMA device lists, max direct I/O
   size, and cache sizes for your hardware; decide explicitly whether
   `allow_compat_mode` should be on in production.


## Common Pitfalls
<a id="pitfalls"></a>

### What to Avoid When Using GPUDirect Storage

1. **Silently running in compatibility mode** — code "works" but routes through a
   CPU bounce buffer, so you see no speedup. *Avoid by* checking `gdscheck -p`
   and `gds_stats` for the count of true-GDS vs. compat-mode I/O.
2. **Unaligned I/O** — non-4 KiB offsets/sizes or buffers force fallback or
   errors. *Avoid by* sizing transfers to alignment boundaries and letting a
   wrapper (kvikio/DALI) manage O_DIRECT.
3. **Tiny random reads** — DMA setup overhead dominates; throughput is worse than
   buffered I/O. *Avoid by* batching, coalescing, or using a read-ahead strategy.
4. **Unsupported filesystem / device** — using GDS on a filesystem or block
   device that lacks DMA support. *Avoid by* consulting the GDS support matrix and
   `gdscheck` filesystem report before deploying.
5. **Page cache confusion** — O_DIRECT bypasses the page cache, so warm-cache
   microbenchmarks mislead. *Avoid by* benchmarking cold and at realistic dataset
   sizes that exceed RAM.
6. **Driver/module version skew** — `nvidia-fs` not rebuilt after a kernel
   upgrade, or `libcufile` mismatched with the driver. *Avoid by* using DKMS and
   keeping the toolkit, driver, and module versions aligned.


## Performance Optimization
<a id="performance"></a>

### Optimizing GPUDirect Storage for Production

#### Configuration Tuning

Key `cufile.json` and runtime parameters to tune:

- **`properties.max_direct_io_size_kb`** — largest single DMA transfer; raise it
  (e.g. 1024–16384) to reduce per-I/O overhead on big sequential reads.
- **`properties.per_buffer_cache_size_kb` / `properties.io_batchsize`** — control
  internal buffering and batch submission depth for many concurrent I/Os.
- **`properties.allow_compat_mode`** — set `false` in performance testing to make
  unsupported paths fail loudly instead of silently degrading.
- **`properties.rdma_dev_addr_list` / `rdma_load_balancing_policy`** — for remote
  storage, list the RDMA NICs and choose a balancing policy across them.
- **Thread / queue depth & GPU buffer pool size** — enough in-flight I/Os and
  registered buffers to keep the DMA engines and drives busy.

`gdsio` (in `/usr/local/cuda/gds/tools/`) is the reference micro-benchmark for
sweeping these parameters and finding the bandwidth ceiling of your topology.


In [ ]:
# Benchmark GDS with the bundled gdsio tool and verify the true-GDS path.
# Run on a GDS-capable host (shell, not Colab).
#
# gdsio flags: -d GPU -w threads -s size -i io_size -x xfer_type -I rw_mode
#   -x 0 = GPUDirect (cuFile) ; -x 1 = CPU-only (POSIX), for A/B comparison
#
# !/usr/local/cuda/gds/tools/gdsio -f /mnt/nvme/gdsio.dat \
#     -d 0 -w 8 -s 8G -i 1M -x 0 -I 1     # write via GDS, 8 threads, 1MiB I/Os
# !/usr/local/cuda/gds/tools/gdsio -f /mnt/nvme/gdsio.dat \
#     -d 0 -w 8 -s 8G -i 1M -x 0 -I 0     # read  via GDS
#
# Compare -x 0 (GDS) against -x 1 (POSIX) to quantify the speedup on YOUR topology.
print("Use gdsio -x 0 vs -x 1 to A/B GPUDirect against the CPU bounce-buffer path.")


## Production Deployment
<a id="deployment"></a>

### Deploying GPUDirect Storage in Production

#### Docker Deployment

GDS in containers requires the **NVIDIA Container Toolkit** plus exposing the
`nvidia-fs` character devices and a GDS-capable mount. Use a CUDA base image that
includes `libcufile`.

```dockerfile
FROM nvcr.io/nvidia/cuda:12.4.1-devel-ubuntu22.04
# libcufile ships in the CUDA image; add kvikio for Python GDS
RUN pip install --no-cache-dir kvikio-cu12 cupy-cuda12x
WORKDIR /app
COPY . /app
CMD ["python3", "train.py"]
```

```bash
# Run with GPU + GDS device access. nvidia-fs must be loaded on the HOST.
docker run --gpus all \
  --device /dev/nvidia-fs0 --device /dev/nvidia-fs1 \
  -v /etc/cufile.json:/etc/cufile.json:ro \
  -v /mnt/nvme:/mnt/nvme \
  my-gds-app:latest
```

#### Kubernetes Deployment

Schedule onto GPU nodes (via the NVIDIA device plugin) and mount the GDS host
paths / `nvidia-fs` devices. NVIDIA's GDS support is exposed through the GPU
Operator on supported nodes.

```yaml
apiVersion: v1
kind: Pod
metadata:
  name: gds-trainer
spec:
  containers:
    - name: trainer
      image: my-gds-app:latest
      resources:
        limits:
          nvidia.com/gpu: 1        # NVIDIA device plugin allocates the GPU
      volumeMounts:
        - { name: cufile-config, mountPath: /etc/cufile.json, subPath: cufile.json }
        - { name: nvme, mountPath: /mnt/nvme }
        - { name: nvidia-fs, mountPath: /dev/nvidia-fs0 }
  volumes:
    - name: cufile-config
      configMap: { name: cufile-config }
    - name: nvme
      hostPath: { path: /mnt/nvme, type: Directory }
    - name: nvidia-fs
      hostPath: { path: /dev/nvidia-fs0, type: CharDevice }
```


## Monitoring and Observability
<a id="monitoring"></a>

### Monitoring GPUDirect Storage in Production

#### Key Metrics to Track

- **GDS vs. compat-mode I/O counts** — the share of I/O taking the true DMA path
  vs. falling back to the CPU bounce buffer (the #1 health signal).
- **Read/write bandwidth (GiB/s) and IOPS** — sustained throughput per GPU and
  aggregate, compared against your topology's PCIe/NVMe ceiling.
- **Average I/O size and latency** — to spot small-random patterns that defeat
  GDS, and tail latency from the storage backend.
- **Error / fallback counters** — alignment failures, RDMA errors, registration
  failures surfaced by `nvidia-fs`.

#### Tools

- **`gds_stats -p <pid> -l 3`** — per-process live cuFile counters (GDS vs.
  compat, bytes, I/O sizes). Requires `cufile.json` logging enabled.
- **`/proc/driver/nvidia-fs/stats`** — kernel-module-wide counters for DMA ops,
  errors, and active mappings.
- **`gdscheck.py -p`** — periodic platform-health re-check after kernel/driver
  changes.

```bash
# Live per-process cuFile stats while the workload runs
gds_stats -p $(pgrep -f train.py) -l 3
# Kernel module counters (errors, ops, mappings)
cat /proc/driver/nvidia-fs/stats
```

#### Logging Best Practices

- Enable `cufile.json` logging at an appropriate level (raise to `INFO`/`DEBUG`
  only while diagnosing — `TRACE` is expensive).
- Alert on a rising compat-mode ratio or fallback-error counters.
- Correlate GDS bandwidth with GPU utilization to confirm I/O isn't starving
  compute.


## Troubleshooting
<a id="troubleshooting"></a>

### Common Issues with GPUDirect Storage

#### Issue 1: All I/O runs in compatibility mode (no speedup)

**Symptoms**: `gds_stats` shows ~100% compat-mode I/O; GDS bandwidth equals the
plain POSIX path.

**Cause**: Unsupported filesystem/device, missing/unloaded `nvidia-fs`, or
`allow_compat_mode` masking a real failure.

**Solution**: Run `gdscheck.py -p`; confirm the filesystem is *Supported*; check
`lsmod | grep nvidia_fs`; mount on supported storage (NVMe ext4/XFS or a
GDS-aware parallel FS); temporarily set `allow_compat_mode=false` to surface the
underlying error.

#### Issue 2: `cuFileRead`/`cuFileWrite` returns an alignment or I/O error

**Symptoms**: cuFile API returns an error; transfers fail or partially complete.

**Cause**: File not opened with `O_DIRECT`, or offset/size/buffer not 4 KiB
aligned.

**Solution**: Open with `O_DIRECT`, align offsets and sizes to 4 KiB (or
`MAX_DIRECT_IO_SIZE` granularity), and register device buffers; or use a wrapper
(kvikio/DALI) that enforces alignment for you.

#### Issue 3: `nvidia-fs` module won't load after a kernel upgrade

**Symptoms**: `modprobe nvidia-fs` fails; `gdscheck` reports the module missing.

**Cause**: DKMS didn't rebuild `nvidia-fs` against the new kernel, or
driver/toolkit version skew.

**Solution**: Reinstall/rebuild via DKMS (`sudo dkms autoinstall`), reboot, and
ensure the NVIDIA driver, CUDA toolkit, and `nvidia-fs` versions are mutually
compatible.


## Comparison with Alternatives
<a id="comparison"></a>

### How GPUDirect Storage Compares to Other Solutions

| Aspect | GPUDirect Storage | Standard POSIX read + `cudaMemcpy` | Host-pinned (`cudaHostRegister`) staging |
|--------|-------------------|------------------------------------|------------------------------------------|
| Data path | Storage/NIC → GPU (1 DMA hop) | Storage → page cache → user buf → GPU | Storage → pinned host buf → GPU |
| CPU involvement in copy | None (CPU only issues I/O) | High (kernel + user copies) | Moderate (still copies into host RAM) |
| System-memory bandwidth used | None for the payload | Full payload twice | Full payload once |
| Peak bandwidth at scale | Highest (scales with drives/GPUs) | Limited by CPU/memcpy | Better than naive, still host-bound |
| Setup complexity | Higher (driver, `nvidia-fs`, alignment) | Lowest | Low–moderate |
| Best for | Large I/O, many GPUs/drives, parallel FS | Small data / prototyping | Medium pipelines without GDS support |

GDS is **complementary** to the broader GPUDirect family: **GPUDirect RDMA**
moves data NIC↔GPU (and underpins GDS over NVMe-oF), while **GPUDirect P2P**
moves data GPU↔GPU. GDS specifically targets the **storage↔GPU** edge.

### When to Choose This Tool

Choose GPUDirect Storage when:

- I/O is on the critical path and the dataset exceeds host RAM / page cache.
- Transfers are large and you can keep them aligned and sequential-ish.
- You run many GPUs and/or drives, or read from a parallel FS over RDMA.
- The CPU is a bottleneck doing storage `memcpy` that you'd rather spend on
  preprocessing.

Stick with standard POSIX + `cudaMemcpy` for small data, prototyping, or when the
working set comfortably fits in cache.


## Resources
<a id="resources"></a>

### Official Documentation

- GPUDirect Storage overview: https://developer.nvidia.com/gpudirect-storage
- cuFile API / GDS docs: https://docs.nvidia.com/gpudirect-storage/
- GDS troubleshooting & install guide: https://docs.nvidia.com/gpudirect-storage/troubleshooting-guide/
- cuFile API reference: https://docs.nvidia.com/gpudirect-storage/api-reference-guide/

### Tutorials and Guides

- NVIDIA blog — "GPUDirect Storage: A Direct Path Between Storage and GPU Memory": https://developer.nvidia.com/blog/gpudirect-storage/
- Magnum IO developer hub: https://developer.nvidia.com/magnum-io
- NVIDIA DALI (GDS-backed data loading): https://docs.nvidia.com/deeplearning/dali/

### Community Resources

- kvikio (RAPIDS Python cuFile bindings): https://github.com/rapidsai/kvikio
- NVIDIA Developer Forums — Storage: https://forums.developer.nvidia.com/
- RAPIDS community: https://rapids.ai/

### Related Technologies

- GPUDirect RDMA (NIC ↔ GPU)
- GPUDirect P2P (GPU ↔ GPU)
- NVIDIA Magnum IO (the umbrella I/O stack GDS belongs to)
- NVMe-oF over RDMA (RoCE / InfiniBand) and parallel filesystems (Lustre/EXAScaler, GPFS, WekaFS, VAST, BeeGFS)
